<a href="https://colab.research.google.com/github/Dongye-L/Smart-ball/blob/main/ball_detect_Yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Tue Sep 22 18:06:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# 先挂载 Drive（每次运行时重置后都要跑）
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# 从 Drive 复制一份到本地
!cp -r "/content/drive/MyDrive/Yolo-ball/custom_data" /content/custom_data

# 下载脚本
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/imIron-man/Train-and-Deploy-YOLO-Models/main/Train-and-Deploy-YOLO-Models-main/utils/train_val_split.py

# 在本地划分
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

In [ ]:
!ls /content/custom_data

In [11]:
!ls -lh

total 398M
-rw-r--r-- 1 root root 398M Sep 22 18:28 project-ball.zip
drwxr-xr-x 1 root root 4.0K Sep  4 13:32 sample_data


In [ ]:
!unzip -FF /content/project-ball.zip -d /content/custom_data/

Archive:  /content/project-ball.zip
   creating: /content/custom_data/images/
   creating: /content/custom_data/labels/
  inflating: /content/custom_data/classes.txt  
  inflating: /content/custom_data/notes.json  
  inflating: /content/custom_data/images/0360a3a2-sample_01_130.900s.png  
  inflating: /content/custom_data/images/06665d23-sample_20_45.060s.png  
  inflating: /content/custom_data/images/0cb1d446-sample_19_132.037s.png  
  inflating: /content/custom_data/images/0f236a5b-sample_10_44.528s.png  
  inflating: /content/custom_data/images/11fbc1e6-sample_14_177.857s.png  
  inflating: /content/custom_data/images/120cc357-sample_20_355.240s.png  
  inflating: /content/custom_data/images/12999fcb-sample_01_84.350s.png  
  inflating: /content/custom_data/images/1eeb8707-sample_03_44.156s.png  
  inflating: /content/custom_data/images/1f7ce73e-sample_20_85.340s.png  
  inflating: /content/custom_data/images/21527d9c-sample_16_85.132s.png  
  inflating: /content/custom_data/images/

In [12]:
!unzip -o /content/project-ball.zip -d /content/custom_data

Archive:  /content/project-ball.zip
   creating: /content/custom_data/images/
   creating: /content/custom_data/labels/
  inflating: /content/custom_data/classes.txt  
  inflating: /content/custom_data/notes.json  
  inflating: /content/custom_data/images/0360a3a2-sample_01_130.900s.png  
  inflating: /content/custom_data/images/06665d23-sample_20_45.060s.png  
  inflating: /content/custom_data/images/0cb1d446-sample_19_132.037s.png  
  inflating: /content/custom_data/images/0f236a5b-sample_10_44.528s.png  
  inflating: /content/custom_data/images/11fbc1e6-sample_14_177.857s.png  
  inflating: /content/custom_data/images/120cc357-sample_20_355.240s.png  
  inflating: /content/custom_data/images/12999fcb-sample_01_84.350s.png  
  inflating: /content/custom_data/images/1eeb8707-sample_03_44.156s.png  
  inflating: /content/custom_data/images/1f7ce73e-sample_20_85.340s.png  
  inflating: /content/custom_data/images/21527d9c-sample_16_85.132s.png  
  inflating: /content/custom_data/images/

划分训练集和验证集

In [13]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/imIron-man/Train-and-Deploy-YOLO-Models/main/Train-and-Deploy-YOLO-Models-main/utils/train_val_split.py
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

--2026-09-22 18:30:34--  https://raw.githubusercontent.com/imIron-man/Train-and-Deploy-YOLO-Models/main/Train-and-Deploy-YOLO-Models-main/utils/train_val_split.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3203 (3.1K) [text/plain]
Saving to: ‘/content/train_val_split.py’

/content/train_val_ 100%[===================>]   3.13K  --.-KB/s    in 0s      

2026-09-22 18:30:35 (52.9 MB/s) - ‘/content/train_val_split.py’ saved [3203/3203]

Created folder at /content/data/train/images.
Created folder at /content/data/train/labels.
Created folder at /content/data/validation/images.
Created folder at /content/data/validation/labels.
Number of image files: 105
Number of annotation files: 105
Images moving to train: 94
Images moving to validation: 11


3.安装Ultralytics

In [14]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 9.2 MB/s eta 0:00:00


生成训练配置文件，系统根据classes.txt自动生成data.yaml配置文件，为下一步YOLO模型训练做好准备

In [15]:
# 下面的 Python 代码会自动创建 YOLO 训练所需的 data.yaml 配置文件。
# 读取 classes.txt 文件中的类别名称。
# 自动获取类别数量 (nc)。
# 生成训练集和验证集路径配置。
# 按照 Ultralytics YOLO 要求写入 data.yaml 文件。

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

    # 检查 classes.txt 是否存在
    if not os.path.exists(path_to_classes_txt):
        print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
        return

    # 读取 classes.txt 中的类别名称
    with open(path_to_classes_txt, 'r') as f:
        classes = []
        for line in f.readlines():
            if len(line.strip()) == 0:
                continue
            classes.append(line.strip())
    number_of_classes = len(classes)

    # 构建 data.yaml 的内容
    data = {
        'path': '/content/data',
        'train': 'train/images',
        'val': 'validation/images',
        'nc': number_of_classes,
        'names': classes
    }

    # 写入 data.yaml
    with open(path_to_data_yaml, 'w') as f:
        yaml.dump(data, f, sort_keys=False)

    print(f'Created config file at {path_to_data_yaml}')

    return

path_to_classes_txt='/content/custom_data/classes.txt'
path_to_data_yaml='/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml



Created config file at /content/data.yaml

File contents:

path: /content/data
train: train/images
val: validation/images
nc: 1
names:
- ball


5.训练模型

In [16]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=60 imgsz=640

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.159 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None

6.测试模型

In [18]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=data/validation/images save=True

Ultralytics 8.4.159 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs

image 1/11 /content/data/validation/images/21527d9c-sample_16_85.132s.png: 384x640 1 ball, 10.9ms
image 2/11 /content/data/validation/images/45c26421-sample_11_84.871s.png: 384x640 1 ball, 20.6ms
image 3/11 /content/data/validation/images/4aa959d0-sample_03_177.301s.png: 384x640 1 ball, 12.8ms
image 4/11 /content/data/validation/images/4bdbd321-sample_11_44.582s.png: 384x640 1 ball, 14.7ms
image 5/11 /content/data/validation/images/4bf63fd9-sample_18_355.120s.png: 384x640 1 ball, 13.0ms
image 6/11 /content/data/validation/images/5f9912a7-sample_14_44.741s.png: 384x640 1 ball, 11.5ms
image 7/11 /content/data/validation/images/76498229-sample_05_131.153s.png: 384x640 1 ball, 11.3ms
image 8/11 /content/data/validation/images/a65a256a-sample_19_178.109s.png: 384x640 1 ball, 11.3ms
image 9/11 /content/data/validation/images/

In [19]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/detect/predic/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')

把runs打压缩包，方便下载

In [20]:
!zip -r /content/runs.zip /content/runs

  adding: content/runs/ (stored 0%)
  adding: content/runs/detect/ (stored 0%)
  adding: content/runs/detect/predict/ (stored 0%)
  adding: content/runs/detect/predict/ee6e9229-sample_16_177.958s.jpg (deflated 0%)
  adding: content/runs/detect/predict/4bf63fd9-sample_18_355.120s.jpg (deflated 0%)
  adding: content/runs/detect/predict/4bdbd321-sample_11_44.582s.jpg (deflated 0%)
  adding: content/runs/detect/predict/21527d9c-sample_16_85.132s.jpg (deflated 0%)
  adding: content/runs/detect/predict/45c26421-sample_11_84.871s.jpg (deflated 0%)
  adding: content/runs/detect/predict/76498229-sample_05_131.153s.jpg (deflated 0%)
  adding: content/runs/detect/predict/4aa959d0-sample_03_177.301s.jpg (deflated 0%)
  adding: content/runs/detect/predict/5f9912a7-sample_14_44.741s.jpg (deflated 0%)
  adding: content/runs/detect/predict/b34bcfe3-sample_18_44.954s.jpg (deflated 0%)
  adding: content/runs/detect/predict/cb0cc183-sample_15_131.784s.jpg (deflated 0%)
  adding: content/runs/detect/predi